# Apriori

**The big picture:** the data is **7,501 shopping baskets** from a shop over one week. Each basket is just a list of items one customer bought. The goal is to find rules like **"people who buy pasta also tend to buy escalope"**, so the shop can place those items together or offer deals on them.

This is **unsupervised** learning: there is no "right answer" column. The computer just looks for patterns in what is bought together.

The core idea of Apriori: if two items are rarely bought together, any bigger set that contains them must be even rarer. So it can throw those away early and only check combinations that are common enough. ("A priori" is Latin for "from what is already known".)

Each rule gets three scores, explained in the training step: **support, confidence and lift**.

### Before you start
- **Read first:** **k_means_clustering** (the idea of unsupervised learning, with no answer column) and **data_preprocessing_tools** (reading a CSV with pandas); the rest of Apriori is new.
- **New words in this notebook:** *association rule*, *support*, *confidence*, *lift*, *transaction*, *apyori*. Look them up in the [glossary](../../../GLOSSARY.md) if you get stuck.

## Importing the libraries

- `!pip install apyori` installs a small add-on library called **apyori**, which does the rule-finding. The `!` means "run this as a command, not as Python". The output just says it was already installed. You only need to do this once per computer.
- **pandas** reads and handles tables, like Excel in Python. It reads the CSV and shows the final rules as a neat table.
- **numpy** and **matplotlib** are imported out of habit but not really used: the results here are a table, not a chart.

In [0]:
!pip install apyori

In [0]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Data Preprocessing

- `header = None` tells pandas the file has **no title row**: the very first line is already a shopping basket.
- Each of the **7,501 rows** is one customer's basket from a week at a shop, and each row lists **up to 20 items** (e.g. *burgers, meatballs, eggs*).
- The apyori library doesn't want a table. It wants a **list of baskets**, where each basket is a list of words. The loop builds exactly that:
  - `for i in range(0, 7501)` goes through every basket.
  - `for j in range(0, 20)` goes through the 20 item slots in that basket.
  - `str(...)` turns each item into text.

**A small quirk:** short baskets have empty slots. pandas fills them with a "missing" value, and `str()` turns that into the word `'nan'`. So most baskets contain a fake item called `'nan'`. It doesn't affect the results here, because the thresholds in the next step filter it out, but it is good to know.

In [0]:
dataset = pd.read_csv('Market_Basket_Optimisation.csv', header = None)
transactions = []
for i in range(0, 7501):
  transactions.append([str(dataset.values[i,j]) for j in range(0, 20)])

## Training the Apriori model on the dataset

A rule looks like **"if a basket has A (left side), it probably also has B (right side)"**. Each argument sets a minimum bar a rule must clear:

- `min_support = 0.003` (**support** = how often a pair appears, out of all baskets). 0.003 is 0.3% of 7,501 baskets, which is about **23 baskets**. **Why this number?** The idea is "a product bought about 3 times a day, over a 7-day week": 3 × 7 / 7,501 ≈ 0.003. Anything rarer is ignored as too rare to matter.
- `min_confidence = 0.2` (**confidence** = out of the baskets that have A, how many also have B). 0.2 means "when someone buys A, they also buy B **at least 20% of the time**". **Why not higher?** Set it too high (e.g. 0.8) and almost no rules survive, or only obvious ones.
- `min_lift = 3` (**lift** = how much more likely B is when A is in the basket, compared with normal). Lift 1 means A makes no difference. Lift 3 means "buying A makes buying B **3 times more likely** than usual". **Why it matters:** a very popular item like mineral water turns up with everything, so it has high confidence with almost anything. Lift removes that effect and keeps only real links.
- `max_length = 2` caps rules at **two items**: one on the left, one on the right. (`min_length = 2` is there too, but apyori actually ignores it; the `min_lift = 3` filter is what removes the single-item records.) **Why?** "Buy one, get the other" deals are about pairs, and pairs are easy to read.

`rules` isn't a list yet. It is a "to-do" object that does the work when you ask for the results (next step).

In [0]:
from apyori import apriori
rules = apriori(transactions = transactions, min_support = 0.003, min_confidence = 0.2, min_lift = 3, min_length = 2, max_length = 2)

## Visualising the results

There is no chart in this notebook. "Visualising" here means turning the raw output into a readable table and sorting it so the best rules come first.

### Displaying the first results coming directly from the output of the apriori function

- `list(rules)` makes apyori actually do the work and collect every rule that passed all the thresholds.
- Typing `results` on its own prints them. The raw output is hard to read. Each `RelationRecord` holds:
  - `items`: the pair, e.g. `{'chicken', 'light cream'}`;
  - `support`: how often the pair appears;
  - `ordered_statistics`: the rule's direction (`items_base` = left side, `items_add` = right side) with its `confidence` and `lift`.

Only **9 rules** pass all the thresholds.

In [0]:
results = list(rules)

In [0]:
results

[RelationRecord(items=frozenset({'chicken', 'light cream'}), support=0.004532728969470737, ordered_statistics=[OrderedStatistic(items_base=frozenset({'light cream'}), items_add=frozenset({'chicken'}), confidence=0.29059829059829057, lift=4.84395061728395)]),
 RelationRecord(items=frozenset({'mushroom cream sauce', 'escalope'}), support=0.005732568990801226, ordered_statistics=[OrderedStatistic(items_base=frozenset({'mushroom cream sauce'}), items_add=frozenset({'escalope'}), confidence=0.3006993006993007, lift=3.790832696715049)]),
 RelationRecord(items=frozenset({'pasta', 'escalope'}), support=0.005865884548726837, ordered_statistics=[OrderedStatistic(items_base=frozenset({'pasta'}), items_add=frozenset({'escalope'}), confidence=0.3728813559322034, lift=4.700811850163794)]),
 RelationRecord(items=frozenset({'honey', 'fromage blanc'}), support=0.003332888948140248, ordered_statistics=[OrderedStatistic(items_base=frozenset({'fromage blanc'}), items_add=frozenset({'honey'}), confidence=0

### Putting the results well organised into a Pandas DataFrame

- The `inspect` function digs into each messy record and pulls out five things:
  - `lhs` = **Left Hand Side** (the "if they buy this..." item);
  - `rhs` = **Right Hand Side** (the "...they also buy this" item);
  - `supports`, `confidences`, `lifts` = the three scores.
- The `result[2][0][...]` parts just mean "go to the 3rd field, take its first entry, then pick out one piece". It's unpacking, not maths.
- `zip` lines the five lists up row by row, and `pd.DataFrame` turns them into a table with named columns.

In [0]:
def inspect(results):
    lhs         = [tuple(result[2][0][0])[0] for result in results]
    rhs         = [tuple(result[2][0][1])[0] for result in results]
    supports    = [result[1] for result in results]
    confidences = [result[2][0][2] for result in results]
    lifts       = [result[2][0][3] for result in results]
    return list(zip(lhs, rhs, supports, confidences, lifts))
resultsinDataFrame = pd.DataFrame(inspect(results), columns = ['Left Hand Side', 'Right Hand Side', 'Support', 'Confidence', 'Lift'])

### Displaying the results non sorted

This shows the 9 rules in the order apyori found them, which has no particular meaning.

**How to read one row:** take row 0, *light cream → chicken*, with support 0.0045, confidence 0.29 and lift 4.84:
- **Support 0.0045:** about 0.45% of all baskets (roughly 34 of 7,501) contain both.
- **Confidence 0.29:** of the customers who bought light cream, 29% also bought chicken.
- **Lift 4.84:** buying light cream makes buying chicken almost **5 times more likely** than for a random customer.

In [0]:
resultsinDataFrame

,Left Hand Side,Right Hand Side,Support,Confidence,Lift
0,light cream,chicken,0.004533,0.290598,4.843951
1,mushroom cream sauce,escalope,0.005733,0.300699,3.790833
2,pasta,escalope,0.005866,0.372881,4.700812
3,fromage blanc,honey,0.003333,0.245098,5.164271
4,herb & pepper,ground beef,0.015998,0.323450,3.291994
5,tomato sauce,ground beef,0.005333,0.377358,3.840659
6,light cream,olive oil,0.003200,0.205128,3.114710
7,whole wheat pasta,olive oil,0.007999,0.271493,4.122410
8,pasta,shrimp,0.005066,0.322034,4.506672


### Displaying the results sorted by descending lifts

- `nlargest(n = 10, columns = 'Lift')` sorts by lift, highest first, and keeps the top 10. There are only 9 rules, so all of them are shown.
- **Why sort by lift?** Lift is the best single measure of how *strong* the link is.
- The top rule is **fromage blanc → honey** (lift 5.16), followed by **light cream → chicken** (4.84) and **pasta → escalope** (4.70).
- Note **herb & pepper → ground beef**: its lift (3.29) is near the bottom, but its support (0.016) is by far the highest, so it affects the most customers.

What "good" looks like: high lift (a strong link) together with decent support (it happens often enough to be worth acting on).

**In one sentence:** turn 7,501 shopping baskets into lists, let Apriori find pairs of items that are bought together often enough (support), reliably enough (confidence) and far more than chance (lift), then show those rules in a table sorted by how strong the link is.

In [0]:
resultsinDataFrame.nlargest(n = 10, columns = 'Lift')

,Left Hand Side,Right Hand Side,Support,Confidence,Lift
3,fromage blanc,honey,0.003333,0.245098,5.164271
0,light cream,chicken,0.004533,0.290598,4.843951
2,pasta,escalope,0.005866,0.372881,4.700812
8,pasta,shrimp,0.005066,0.322034,4.506672
7,whole wheat pasta,olive oil,0.007999,0.271493,4.122410
5,tomato sauce,ground beef,0.005333,0.377358,3.840659
1,mushroom cream sauce,escalope,0.005733,0.300699,3.790833
4,herb & pepper,ground beef,0.015998,0.323450,3.291994
6,light cream,olive oil,0.003200,0.205128,3.114710


## Common mistakes to avoid
- **Forgetting `header = None`.** Without it, pandas treats the first basket as column names, so that basket is lost and the loop over 7,501 rows runs one past the end of the data and crashes.
- **Sorting by confidence instead of lift.** A very popular item (like mineral water) shows up in so many baskets that almost any rule pointing to it has high confidence. Lift corrects for popularity, so sort by lift to find real links.
- **Thinking `min_length = 2` is what keeps single items out.** apyori does not actually use `min_length`; it is `max_length = 2` plus the `min_lift = 3` filter that leave only pairs. If you remove the lift filter, single-item records can slip in and the `inspect` function crashes with an IndexError.
- **Raising `max_length` and trusting the table.** The `inspect` function only takes the first item on each side (`tuple(...)[0]`) and the first rule of each record (`result[2][0]`), so with 3-item rules the table quietly hides items. Look at the raw `results` when you change rule length.

## Try this yourself
Change one thing, rerun the notebook, and see what happens.

1. In the training cell, change `min_support = 0.003` to `min_support = 0.005` and re-run every cell below it. What to notice: only the rules whose Support was already 0.005 or more in the table survive (you can predict which ones before running it), because their confidence and lift do not change.
2. In the training cell, change `min_confidence = 0.2` to `min_confidence = 0.35` and re-run the cells below. What to notice: only the rules whose Confidence was 0.35 or more are left (pasta → escalope and tomato sauce → ground beef), showing how quickly a high bar removes rules.
3. In the last cell, change `nlargest(n = 10, columns = 'Lift')` to `columns = 'Support'`. What to notice: herb & pepper → ground beef jumps from near the bottom to the top. A strong link (lift) and a common pair (support) are different things.
4. In a new cell, run `print(sum('nan' in t for t in transactions))`. What to notice: most baskets contain the fake `'nan'` item made from empty slots, yet it never appears in the rules. Because it is in almost every basket, it cannot make any other item noticeably more likely, so its lift stays far below 3.

## Check your understanding
Try to answer in your head first, then click to reveal.

**1. Why does the notebook use lift and not just confidence?**

<details><summary>Show answer</summary>

Confidence can be high simply because the second item is very popular. Lift compares the chance of buying B after A with the normal chance of buying B, so it only rewards real links. A lift of 3 means B is 3 times more likely than usual.

</details>

**2. Read the sorted table: which rule has the strongest link, and what does its lift mean?**

<details><summary>Show answer</summary>

fromage blanc → honey, with lift about 5.16. Customers who buy fromage blanc are about 5 times more likely to buy honey than a random customer.

</details>

**3. In the light cream → chicken row, what does the support of about 0.0045 mean in plain words?**

<details><summary>Show answer</summary>

About 0.45% of all 7,501 baskets (roughly 34 baskets) contain both light cream and chicken.

</details>

**4. What would happen if you set `min_support = 0.02`?**

<details><summary>Show answer</summary>

No rules would come out at all. The highest support in the table is about 0.016 (herb & pepper and ground beef), so every pair would fall below the bar and the table would be empty.

</details>

---
📚 [Learning path](../../../LEARNING_PATH.md) · [Glossary](../../../GLOSSARY.md)